In [0]:
# MAGIC %md
# ============================================================
# NOTEBOOK   : gold_review_analysis
# PURPOSE    : Customer satisfaction analysis by category
# SOURCE     : silver.order_reviews, silver.order_items,
#              silver.products
# DESTINATION: fincompliance_catalog.gold.review_analysis
# METRICS:
#   - Average review score by category
#   - Sentiment distribution by category
#   - Comment rate by category
#   - Categories with most negative reviews
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    avg,
    sum as spark_sum,
    round as spark_round,
    when,
    current_timestamp
)

In [0]:
# ============================================================
# READ FROM SILVER
# ============================================================

df_reviews = spark.table(f"{SILVER_DB}.order_reviews")
df_order_items = spark.table(f"{SILVER_DB}.order_items")
df_products = spark.table(f"{SILVER_DB}.products")

print("=" * 60)
print("SILVER TABLES LOADED")
print("=" * 60)
print(f"order_reviews : {df_reviews.count():,} rows")
print(f"order_items   : {df_order_items.count():,} rows")
print(f"products      : {df_products.count():,} rows")

print("\nReview sentiment breakdown:")
df_reviews \
    .groupBy("review_sentiment") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

print("\nReview score distribution:")
df_reviews \
    .groupBy("review_score") \
    .count() \
    .orderBy("review_score") \
    .show(truncate=False)

In [0]:
# ============================================================
# CALCULATE REVIEW ANALYSIS BY PRODUCT CATEGORY
# Join reviews with order_items and products
# Group by English category name
# Filter out unscored reviews for accurate averages
# ============================================================

df_review_analysis = (
    df_reviews
    .filter(col("review_score").isNotNull())
    .join(
        df_order_items.select("order_id", "product_id"),
        on="order_id",
        how="left"
    )
    .join(
        df_products.select(
            "product_id",
            "product_category_name_english"
        ),
        on="product_id",
        how="left"
    )
    .groupBy("product_category_name_english")
    .agg(
        count("review_id").alias("total_reviews"),
        spark_round(avg("review_score"), 2)
            .alias("avg_review_score"),
        spark_sum(
            when(col("review_score") == 5, 1)
            .otherwise(0)
        ).alias("five_star_count"),
        spark_sum(
            when(col("review_score") == 1, 1)
            .otherwise(0)
        ).alias("one_star_count"),
        spark_sum(
            when(col("review_sentiment") == "positive", 1)
            .otherwise(0)
        ).alias("positive_count"),
        spark_sum(
            when(col("review_sentiment") == "negative", 1)
            .otherwise(0)
        ).alias("negative_count"),
        spark_sum(
            when(col("review_sentiment") == "neutral", 1)
            .otherwise(0)
        ).alias("neutral_count"),
        spark_sum("has_comment").alias("total_comments")
    )
    .withColumn(
        "five_star_rate_pct",
        spark_round(
            (col("five_star_count") /
             col("total_reviews")) * 100, 2
        )
    )
    .withColumn(
        "one_star_rate_pct",
        spark_round(
            (col("one_star_count") /
             col("total_reviews")) * 100, 2
        )
    )
    .withColumn(
        "positive_rate_pct",
        spark_round(
            (col("positive_count") /
             col("total_reviews")) * 100, 2
        )
    )
    .withColumn(
        "comment_rate_pct",
        spark_round(
            (col("total_comments") /
             col("total_reviews")) * 100, 2
        )
    )
    .orderBy(col("avg_review_score").desc())
)

print("Top 10 categories by review score:")
df_review_analysis \
    .select(
        "product_category_name_english",
        "total_reviews",
        "avg_review_score",
        "five_star_rate_pct",
        "one_star_rate_pct",
        "positive_rate_pct"
    ) \
    .show(10, truncate=False)

print("\nBottom 10 categories by review score:")
df_review_analysis \
    .select(
        "product_category_name_english",
        "total_reviews",
        "avg_review_score",
        "five_star_rate_pct",
        "one_star_rate_pct",
        "positive_rate_pct"
    ) \
    .orderBy(col("avg_review_score").asc()) \
    .show(10, truncate=False)

print(f"\nTotal categories : {df_review_analysis.count()}")

In [0]:
# ============================================================
# ADD GOLD AUDIT COLUMN
# ============================================================

df_review_analysis = df_review_analysis \
    .withColumn("gold_updated_at", current_timestamp())

print("Final columns:")
for col_name in df_review_analysis.columns:
    print(f"  {col_name}")
print(f"Total categories : {df_review_analysis.count()}")

In [0]:
# ============================================================
# WRITE TO GOLD
# ============================================================

(
    df_review_analysis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.review_analysis")
)

print(f"Written to {GOLD_DB}.review_analysis")
print(f"Rows : {df_review_analysis.count()}")

In [0]:
# ============================================================
# VERIFICATION
# ============================================================
print("=" * 60)
print("GOLD.REVIEW_ANALYSIS VERIFICATION")
print("=" * 60)

df_gold = spark.table(f"{GOLD_DB}.review_analysis")
print(f"Total categories : {df_gold.count()}")
print(f"Total columns    : {len(df_gold.columns)}")

print("\nTop 5 categories by review score:")
df_gold \
    .orderBy(col("avg_review_score").desc()) \
    .select(
        "product_category_name_english",
        "total_reviews",
        "avg_review_score",
        "positive_rate_pct"
    ) \
    .show(5, truncate=False)

print("\nBottom 5 categories by review score:")
df_gold \
    .orderBy(col("avg_review_score").asc()) \
    .select(
        "product_category_name_english",
        "total_reviews",
        "avg_review_score",
        "one_star_rate_pct"
    ) \
    .show(5, truncate=False)

print("\nOverall platform average score:")
df_gold.agg(
    spark_round(avg("avg_review_score"), 2)
        .alias("platform_avg_score")
).show()

print("=" * 60)
print("gold.review_analysis verification complete!")
print("=" * 60)